# 4. Execute: actions and state machines

Actions run as control-flow graphs when the model declares successions;
state machines are hierarchical, support `parallel` regions, and have a
simulation clock for `after`/`at` time triggers.

In [ ]:
import sysml2

model = sysml2.loads("""
package Behaviors {
    action def Checkout {
        in itemCount : Integer;
        out total : Real;
        assign total := itemCount * 9.99;
        if total > 50.0 {
            assign total := total * 0.9;    // bulk discount
        }
    }

    action def Pipeline {
        out log : String;
        assign log := "";
        action fetch { assign log := log + "fetch>"; }
        action build { assign log := log + "build>"; }
        action test { assign log := log + "test"; }
        first start then fetch;
        first fetch then build;
        first build then test;
        first test then done;
    }
}
""")
interp = sysml2.Interpreter(model)

## Plain actions: declaration order, traced

In [ ]:
run = interp.run_action("Behaviors::Checkout", inputs={"itemCount": 7})
print("outputs:", run.outputs)
for line in run.trace:
    print("  trace:", line)

## Succession-driven flow: the graph, not the source order, decides

In [ ]:
run = interp.run_action("Behaviors::Pipeline")
print(run.outputs["log"])
print([t for t in run.trace if t.startswith("step")])

## Hierarchical state machines

Composite states enter through their own entry transitions; events go to
the innermost active state first; exits cascade innermost-first.

In [ ]:
machine_model = sysml2.loads("""
package Machines {
    state def Player {
        attribute plays : Integer := 0;
        entry; then stopped;

        state stopped;
        transition first stopped accept play then playing;

        state playing {
            entry assign plays := plays + 1;
            entry; then normal;
            state normal;
            transition first normal accept fast then fastForward;
            state fastForward;
            transition first fastForward accept fast then normal;
        }
        transition first playing accept stop then stopped;
    }
}
""")
sim = sysml2.Interpreter(machine_model).simulate(
    "Machines::Player", events=["play", "fast", "fast", "stop", "play"]
)
for step in sim.trace:
    print(" ", step)
print("final:", sim.final_state, "| plays:", sim.env["plays"])

## Time triggers

Numbers in the event list advance the clock; `accept after d` measures
from state entry, `accept at t` is absolute.

In [ ]:
toaster_model = sysml2.loads("""
package Kitchen {
    state def Toaster {
        attribute pops : Integer := 0;
        entry; then idle;
        state idle;
        transition first idle accept press then toasting;
        state toasting;
        transition first toasting accept after 30.0
            do assign pops := pops + 1
            then idle;
    }
}
""")
interp2 = sysml2.Interpreter(toaster_model)
sim = interp2.simulate("Kitchen::Toaster", events=["press", 15.0, 20.0, "press", 31.0])
for step in sim.trace:
    print(" ", step)
print(f"final: {sim.final_state} at t={sim.time}, pops={sim.env['pops']}")